<a href="https://colab.research.google.com/github/AlekhyaGangopadhyay/IEM-IIT_HCI_Project/blob/main/Chebyshev_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import pandas as pd
import os
from scipy.signal import cheby1, filtfilt
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

def chebyshev_eeg_filter(data, fs=250, lowcut=8, highcut=30, order=4, rp=0.5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = cheby1(order, rp, [low, high], btype='band')
    return filtfilt(b, a, data, axis=-1)

def process_eeg_folder(input_folder='', output_folder_parent='Subject1_Chebyshev_Filtered'):

    base_path = Path('/content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4')
    input_folder = base_path / input_folder

    output_base = Path('/content/drive/MyDrive/Human Computer Interface (HCI)/Processed_Filtered_Data/S2S4_Chebyshev_Filtered')
    output_folder = output_base / output_folder_parent

    output_folder.mkdir(parents=True, exist_ok=True)

    print("🔍 Scanning for Excel files...")
    excel_files = []
    for ext in ['*.xlsx', '*.xls']:
        excel_files.extend(list(input_folder.rglob(ext)))
        excel_files.extend(list(base_path.rglob(ext)))

    print(f"✅ Found {len(excel_files)} Excel files:")
    for f in excel_files[:5]:
        print(f"   📄 {f}")
    if len(excel_files) > 5:
        print(f"   ... and {len(excel_files)-5} more")

    if len(excel_files) == 0:
        print("❌ NO EXCEL FILES FOUND! Check your folder structure.")
        print("📁 Available folders/files:")
        for item in base_path.iterdir():
            print(f"   {'📁' if item.is_dir() else '📄'} {item.name}")
        return

    print(f"\n💾 Output folder: {output_folder}")

    for i, file_path in enumerate(excel_files, 1):
        print(f"\n[{i}/{len(excel_files)}] Processing: {file_path.name}")

        try:
            df = pd.read_excel(file_path)
            data_raw = df.values

            print(f"   Shape: {data_raw.shape}")
            print(f"   Raw range: {data_raw.min():.2f} to {data_raw.max():.2f} μV")

            if data_raw.shape[0] > data_raw.shape[1]:
                data_raw = data_raw.T

            data_cheby = chebyshev_eeg_filter(data_raw, fs=250)

            if data_cheby.shape[1] > 250:
                baseline = data_cheby[:, :250].mean(axis=-1, keepdims=True)
                data_clean = data_cheby - baseline
            else:
                data_clean = data_cheby

            print(f"   Clean range: {data_clean.min():.2f} to {data_clean.max():.2f} μV")

            output_filename = f"Cheby_{file_path.stem}.xlsx"
            output_file = output_folder / output_filename
            df_clean = pd.DataFrame(data_clean.T)
            df_clean.to_excel(output_file, index=False)

            print(f"   ✅ Saved: {output_filename}")

        except Exception as e:
            print(f"   ❌ ERROR: {str(e)[:100]}")
            continue

    print(f"\n🎉 BATCH COMPLETE! {len(excel_files)} files processed")
    print(f"📂 All files: {output_folder}")

process_eeg_folder('', 'Subject2_Chebyshev_Filtered')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Scanning for Excel files...
✅ Found 24 Excel files:
   📄 /content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4/ARROW_Backword_2_synthetic_4.xlsx
   📄 /content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4/ARROW_Forward_2_synthetic_4.xlsx
   📄 /content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4/ARROW_Left_2_synthetic_4.xlsx
   📄 /content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4/Right LETTER2_synthetic_4.xlsx
   📄 /content/drive/MyDrive/Human Computer Interface (HCI)/Original Dataset/Subject 2 Synthetic 4/Right WORD2_synthetic_4.xlsx
   ... and 19 more

💾 Output folder: /content/drive/MyDrive/Human Computer Interface (HCI)/Processed_Filtered_Data/S2S4_Chebyshev_Filtered/Subject2_Chebyshev_Filtered

[1